# Step 1 — Data Collection from UniProt

**LB2 Project · Group 7 · Signal Peptide Prediction**

### Objective
This notebook retrieves high-quality positive (SP+) and negative (SP−) protein datasets directly from the UniProtKB REST API.

**Key features:**
- Uses reviewed (Swiss-Prot) entries only
- Restricted to Eukaryotes (taxonomy ID 2759)
- Minimum sequence length of 40 amino acids
- Positive set: experimentally verified signal peptides
- Negative set: no signal peptide but confirmed secretory pathway localization

### Current Dataset (as of retrieval)
| Set       | Total Retrieved | After Filtering | FASTA Sequences |
|-----------|-----------------|-----------------|-----------------|
| Positive  | 2,949           | **2,932**       | positive.fasta  |
| Negative  | 20,615          | **20,615**      | negative.fasta  |

**Note:** UniProt data can change daily. The numbers above reflect the exact dataset used in this run.

In [ ]:
import requests
from requests.adapters import HTTPAdapter, Retry
import json
import re

---
## Cell 1 — Imports and helper functions

This cell imports required libraries and defines the core functions used for:
- Kingdom classification (Metazoa / Viridiplantae / Fungi / Other)
- Positive and negative entry filtering
- JSON → TSV conversion
- Pagination handling via UniProt's "Link" header

In [ ]:
def get_kingdom(entry):
  if "Metazoa" in entry["organism"]["lineage"]:
    k = "Metazoa"
  elif "Viridiplantae" in entry["organism"]["lineage"]:
    k = "Viridiplantae"
  elif "Fungi" in entry["organism"]["lineage"]:
    k = "Fungi"
  else:
    k = "Other"
  return k

def filter_entry_positive(entry):
    try:
        e_pos = int(entry["features"][0]["location"]["end"]["value"])
        assert(entry["features"][0]["description"] == "")
        assert(e_pos > 13)
    except:
        return False
    return True

def filter_entry_negative(entry):
    return True

def json_to_tsv_positive(entry):
    return (entry["primaryAccession"],
            entry["organism"]["scientificName"],
            get_kingdom(entry),
            entry["sequence"]["length"],
            entry["features"][0]["location"]["end"]["value"])

def json_to_tsv_negative(entry):
    try:
        k = [l for l in entry["lineages"] if l["rank"]=="kingdom"][0]["scientificName"]
    except:
        k = "Other"
    tm = False
    for f in entry["features"]:
        if f["type"]=="Transmembrane":
          if re.search("Helical",f["description"]):
            if f["location"]["start"]["value"]<=90:
              tm = True
              break

    return (entry["primaryAccession"],
            entry["organism"]["scientificName"],
            get_kingdom(entry),
            entry["sequence"]["length"],
            tm)

def get_next_link(headers):
    if "Link" in headers:
        match = re_next_link.match(headers["Link"])
        if match:
            return match.group(1)

def get_batch(batch_url):
    while batch_url:
        response = session.get(batch_url)
        response.raise_for_status()
        total = response.headers["x-total-results"]
        yield response, total
        batch_url = get_next_link(response.headers)
        print(batch_url)

def get_dataset(search_url, filter_function, json_to_tsv_function, columns, output_file_name, output_fasta_file_name):
    n_total, n_filtered = 0, 0
    with open(output_file_name, 'w') as ofs:
      print(*columns, sep="\t", file=ofs)
      with open(output_fasta_file_name, 'w') as ofs_fasta:
        for batch, total in get_batch(search_url):
          batch_json = json.loads(batch.text)
          for entry in batch_json["results"]:
            n_total += 1
            if filter_function(entry):
              n_filtered += 1
              fields = json_to_tsv_function(entry)
              print(*fields, sep="\t", file=ofs)
              print(">", entry["primaryAccession"], sep="", file=ofs_fasta)
              print(entry["sequence"]["value"], file=ofs_fasta)
        ofs_fasta.close()
      ofs.close
    print(f"Total: {n_total}, filtered: {n_filtered}")

---
## Cell 2 — UniProt API session setup

Configures a robust HTTP session with:
- Automatic retries (5 attempts) for transient errors
- Proper handling of pagination using `rel="next"` links
- Required for reliable large-scale downloads from UniProt

In [ ]:
re_next_link = re.compile(r'<(.+)>; rel="next"')
retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))


---
## Cell 3 — UniProt search URLs and column definitions

Two carefully crafted REST API queries are used:

**Positive set (SP+)**  
- Experimentally verified signal peptide (`ft_signal_exp:*`)  
- Full-length, reviewed proteins ≥ 40 aa

**Negative set (SP−)**  
- No signal peptide annotation (`NOT ft_signal:*`)  
- Confirmed localization to secretory pathway compartments

In [ ]:
url_positive="https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28reviewed%3Atrue%29+AND+%28taxonomy_id%3A2759%29+AND+%28length%3A%5B40+TO+*%5D%29+AND+%28fragment%3Afalse%29+AND+%28existence%3A1%29+AND+%28ft_signal_exp%3A*%29%29&size=500"
url_negative="https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28reviewed%3Atrue%29+AND+%28taxonomy_id%3A2759%29+AND+%28length%3A%5B40+TO+*%5D%29+AND+%28fragment%3Afalse%29+NOT+%28ft_signal%3A*%29+AND+%28+%28cc_scl_term_exp%3ASL-0091%29+OR+%28cc_scl_term_exp%3ASL-0191%29+OR+%28cc_scl_term_exp%3ASL-0173%29+OR+%28cc_scl_term_exp%3ASL-0209%29+OR+%28cc_scl_term_exp%3ASL-0204%29+OR+%28cc_scl_term_exp%3ASL-0039%29+%29+AND+%28existence%3A1%29%29&size=500"
col_positive = ["Accession", "Organism", "Kingdom", "Sequence length", "SP cleavage"]
col_negative = ["Accession", "Organism", "Kingdom", "Sequence length", "N-term transmembrane"]

---
## Cell 4 — Download positive dataset (SP+)

Retrieves proteins with **experimentally verified signal peptides**.  
Only entries with a confirmed cleavage site > 13 residues are kept.

In [ ]:
get_dataset(url_positive, filter_entry_positive, json_to_tsv_positive, col_positive, "positive.tsv", "positive.fasta")

https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28reviewed%3Atrue%29%20AND%20%28taxonomy_id%3A2759%29%20AND%20%28length%3A%5B40%20TO%20%2A%5D%29%20AND%20%28fragment%3Afalse%29%20AND%20%28existence%3A1%29%20AND%20%28ft_signal_exp%3A%2A%29%29&cursor=c9bacmxsqhkqgdxgnaquhsybhmvuelm5fxu0j&size=500
https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28reviewed%3Atrue%29%20AND%20%28taxonomy_id%3A2759%29%20AND%20%28length%3A%5B40%20TO%20%2A%5D%29%20AND%20%28fragment%3Afalse%29%20AND%20%28existence%3A1%29%20AND%20%28ft_signal_exp%3A%2A%29%29&cursor=28m7xk8oeejl5mhhil7voukl953i9phd2w397hu&size=500
https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28reviewed%3Atrue%29%20AND%20%28taxonomy_id%3A2759%29%20AND%20%28length%3A%5B40%20TO%20%2A%5D%29%20AND%20%28fragment%3Afalse%29%20AND%20%28existence%3A1%29%20AND%20%28ft_signal_exp%3A%2A%29%29&cursor=28ndf240hnvwz75t65klyjzcujbd3y9nn3i5gky&size=500
https://rest.uniprot.org/uniprotkb/search?format=json&query=%28

---
## Cell 5 — Download negative dataset (SP−)

Retrieves high-quality proteins that **do not** have a signal peptide but are localized in the secretory pathway.  
Also flags any N-terminal transmembrane helices (within the first 90 residues).

In [ ]:
get_dataset(url_negative, filter_entry_negative, json_to_tsv_negative, col_negative, "negative.tsv", "negative.fasta")


https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28reviewed%3Atrue%29%20AND%20%28taxonomy_id%3A2759%29%20AND%20%28length%3A%5B40%20TO%20%2A%5D%29%20AND%20%28fragment%3Afalse%29%20NOT%20%28ft_signal%3A%2A%29%20AND%20%28%20%28cc_scl_term_exp%3ASL-0091%29%20OR%20%28cc_scl_term_exp%3ASL-0191%29%20OR%20%28cc_scl_term_exp%3ASL-0173%29%20OR%20%28cc_scl_term_exp%3ASL-0209%29%20OR%20%28cc_scl_term_exp%3ASL-0204%29%20OR%20%28cc_scl_term_exp%3ASL-0039%29%20%29%20AND%20%28existence%3A1%29%29&cursor=c9bacmxsqhkqgdxflkgr8ocjvob9drkxr36lx&size=500
https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28reviewed%3Atrue%29%20AND%20%28taxonomy_id%3A2759%29%20AND%20%28length%3A%5B40%20TO%20%2A%5D%29%20AND%20%28fragment%3Afalse%29%20NOT%20%28ft_signal%3A%2A%29%20AND%20%28%20%28cc_scl_term_exp%3ASL-0091%29%20OR%20%28cc_scl_term_exp%3ASL-0191%29%20OR%20%28cc_scl_term_exp%3ASL-0173%29%20OR%20%28cc_scl_term_exp%3ASL-0209%29%20OR%20%28cc_scl_term_exp%3ASL-0204%29%20OR%20%28cc_scl_term_

In [ ]:
import pandas as pd

negative_set = pd.read_csv("negative.tsv", sep="\t")
positive_set = pd.read_csv("positive.tsv", sep="\t")

print(len(positive_set))
print(len(negative_set))
print(len(negative_set[negative_set["N-term transmembrane"] == True]))


2932
20615
2465


---
## Cell 6 — Final statistics and summary

**Summary of retrieved data:**

- **Positive set**: 2,932 sequences with experimentally verified signal peptides
- **Negative set**: 20,615 sequences (no signal peptide)
- **Negative set with N-terminal transmembrane helix**: 2,465 proteins

All data has been saved as:
- `positive.tsv` / `positive.fasta`
- `negative.tsv` / `negative.fasta`

**Ready for Step 2:** Run `step2_data_preparation.ipynb` next.